### Installation

In [ ]:
import os

CACHE_DIR = "/workspace/.cache/huggingface"
os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_DIR}/datasets"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_DIR}/hub"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_DIR}/transformers"

In [ ]:
!rm -rf ~/.cache/huggingface

In [ ]:
!pip install -qU unsloth
!pip install -qU unsloth_zoo

In [ ]:
!pip install -qU pip setuptools wheel
# !pip install -qU "unsloth[colab-new]"
!pip install -qU huggingface_hub scipy matplotlib timm
!pip install -qU tensorboard==2.15.1
# !pip install -qU librosa soundfile soxr
!pip install -qU typing_extensions

In [ ]:
import os

if "COLAB_" in "".join(os.environ.keys()):
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN is missing. Set Colab Secrets or export HF_TOKEN in your environment.")


### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

base_model_id = "hypaai/Hypa_Llama3.2-8b-SFT-2025-12-20_II-16bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    # "hypaai/Hypa_Llama3.2-8b-SFT-2025-12-10-16bit"
    # "hypaai/Hypa_Llama3.1-8b-SFT-2025-10-25-16bit",
    # "ccibeekeoc42/Llama-3.2-8B-Instruct-bnb-4bit_merged_16bit_finetune_2025-03-07",
    # "unsloth/Meta-Llama-3.1-8B-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)
finetune_model_id = f"Hypa-Llama3.1-8b-SFT-runpod"
model.config.max_position_embeddings = 131072

In [ ]:
print(f"Max position embeddings of the model: {model.config.max_position_embeddings}")

In [ ]:
print("Searching for RoPE and max position related configs:")
for k, v in model.config.to_dict().items():
    if 'rope' in k.lower() or 'max' in k.lower() or 'position' in k.lower():
        print(f"{k}: {v}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 256, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128, 2048, 4096
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 256,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### Data Prep

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# Set the pad token to the eos token (<|end_of_text|>)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# download this model "hypaai/Hypa-Text-10k"
from datasets import load_dataset
dataset = load_dataset("hypaai/Hypa-Text-10k", split="train", token=hf_token)
dataset

In [ ]:
dataset

In [ ]:
# dataset = dataset.shuffle(seed=42).select(range(5_000))
# dataset

In [ ]:
import random

idx = random.randint(0, len(dataset) - 1)
dataset[idx]['messages']

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)
dataset[idx]['messages']

In [ ]:
def formatting_prompts_func(examples):
    """
    Format chat conversations for Llama 3 / 3.1 / 3.3 Instruct models.

    Expected input:
      examples["messages"] = [
        [
          {"role": "system", "content": "..."},
          {"role": "user", "content": "..."},
          {"role": "assistant", "content": "...", "thinking": "..."},
          ...
        ],
        ...
      ]

    Notes:
    - Uses tokenizer.apply_chat_template(...) so Llama's native special tokens
      (<|begin_of_text|>, <|start_header_id|>, <|end_header_id|>, <|eot_id|>)
      are inserted correctly by the tokenizer.
    - Does manually append eos_token afterward.
    - Replaces Gemma-specific thought-channel tags with plain text markers that
      are safe for Llama chat formatting.
    """

    convos = examples["messages"]
    processed_convos = []

    for convo in convos:
        new_convo = []

        # Detect whether this sample contains assistant reasoning
        has_thinking = any(
            msg.get("role") == "assistant"
            and msg.get("thinking") is not None
            and str(msg.get("thinking")).strip()
            and str(msg.get("thinking")).strip().lower() != "none"
            for msg in convo
        )

        for msg in convo:
            new_msg = msg.copy()
            role = new_msg.get("role")
            content = str(new_msg.get("content", ""))
            thinking = new_msg.get("thinking")

            # Keep your original behavior: if thinking exists anywhere in the convo,
            # annotate the system message accordingly.
            if role == "system":
                if has_thinking:
                    # Safer for Llama than Gemma-specific special tokens
                    new_msg["content"] = f"<|think>\n{content}"
                else:
                    new_msg["content"] = content.removesuffix("\n\nTHINKING=None")

            # Inject assistant reasoning before the visible answer
            if (
                role == "assistant"
                and thinking is not None
                and str(thinking).strip()
                and str(thinking).strip().lower() != "none"
            ):
                thinking = str(thinking).strip()
                new_msg["content"] = f"<think>\n{thinking}\n</think>\n{content}"

            # Keep only keys the chat template expects
            new_convo.append({
                "role": new_msg["role"],
                "content": new_msg["content"],
            })

        processed_convos.append(new_convo)

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in processed_convos
    ]

    # Strip BOS — Unsloth / the trainer adds it back during tokenization
    bos = tokenizer.bos_token or "<|begin_of_text|>"
    texts = [text.removeprefix(bos) for text in texts]

    # Append EOS so the model learns to terminate generation
    eos = tokenizer.eos_token or "<|end_of_text|>"
    texts = [text + eos for text in texts]

    return {"text": texts}

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched=True)
dataset[0]

In [ ]:
import random

idx = random.randint(0, len(dataset) - 1)
dataset[idx]['text']

In [ ]:
# Dropping all columns except conversations and text
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in ["messages", "text"]])
dataset

In [ ]:
from datasets import DatasetDict

# Shuuffling the dataset
dataset = dataset.shuffle(seed=420)

# Split dataset into 90% train and 10% test
train_test_split = dataset.train_test_split(test_size=0.000005)

# Assign train and test datasets
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# Check dataset sizes
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(eval_dataset)}")

### Train the model

In [ ]:
import os
from datetime import datetime

now = datetime.now()
current_time = now.strftime("%Y-%m-%d")
repo_id = f"{finetune_model_id}-{current_time}"

In [ ]:
batch_size = 16
gradient_accumulation_steps = 2
num_train_epochs = 1
save_strategy="steps"
eval_strategy="steps"
warmup_steps = 500
learning_rate = 1e-4
# logging_steps = 500
# save_steps=500
# eval_steps=500
logging_steps = 1_000
save_steps=12_000                  # Save every X updates steps
eval_steps=12_000
optim = "adamw_8bit"
weight_decay = 0.01
max_grad_norm = 1.0
lr_scheduler_type = "cosine"
seed = 3407

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = True, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size = batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        num_train_epochs = num_train_epochs, # Set this for 1 full training run.
        save_strategy=save_strategy,
        eval_strategy=eval_strategy,
        warmup_steps = warmup_steps,
        learning_rate = learning_rate,
        logging_steps = logging_steps,
        save_steps=save_steps,                  # Save every X updates steps
        eval_steps=eval_steps,
        save_total_limit=1,
        optim = optim,
        weight_decay = weight_decay,
        max_grad_norm = max_grad_norm,
        lr_scheduler_type = lr_scheduler_type,
        seed = seed,
        output_dir = f"./{repo_id}_LoRAs",
        report_to = "tensorboard", # Use this for WandB etc
        push_to_hub=True,
        hub_token = hf_token,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss", # ['eval_loss', 'eval_bleu', 'eval_rouge1', 'eval_rouge2', 'eval_rougeL', 'eval_perplexity']
        greater_is_better=False
    ),
)

In [ ]:
print(f"Tokenizer EOS token: {tokenizer.eos_token}")

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### Inference

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "system", "content": "You are a helpful and honest multilingual translation expert specialized in translating to Igbo."},
    {"role": "user", "content": 'Translate this to Igbo and Return ONLY THE EXACT TRANSLATION. "I am hungry and I am tired."'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True,
                         temperature = 1, top_k = 3, min_p = 0.1)
tokenizer.batch_decode(outputs)

# [{'content': 'You are a helpful and honest multilingual translation expert specialized in translating to English.',
#   'role': 'system'},
#  {'content': 'Translate this to English and Return ONLY THE EXACT TRANSLATION. "Anye adido, anam utom nnor Google."',
#   'role': 'user'},
#  {'content': 'She later worked as an employee for Google.',
#   'role': 'assistant'}]

In [ ]:
# Using text Streamer
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    # {"role": "system", "content": "You are a helpful and honest multilingual translation expert specialized in translating to Igbo."},
    {"role": "user", "content": 'Translate this to Igbo and Return ONLY THE EXACT TRANSLATION. "Who is in the garden, a little fine girl. Can I come and see her? No, No No, no."'},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 550,
                   use_cache = True, temperature = 0.5, top_k = 3, min_p = 0.1)

In [ ]:
# model.save_pretrained("lora_model")  # Local saving
# tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

### Saving to float16 for VLLM

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if True: model.push_to_hub_merged(f"hypaai/{repo_id}-16bit", tokenizer, save_method = "merged_16bit", token = hf_token)

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged(f"hypaai/Hypa_Llama3.1-SFT-{current_time}-4bit", tokenizer, save_method = "merged_4bit", token = hf_token)

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")